# Week 1: Topological Foundations of Geographic Data

**Learning Objectives:**
1. Understand metric spaces and their application to geographic data
2. Explore compactness in spatial regions (bounded territories)
3. Study connectedness of settlement networks
4. Visualize graphs as topological spaces (nodes, edges, structure)

**Key Insight:** The topology of geographic data determines why neural networks need depth.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import distance_matrix
import networkx as nx

print('Imports successful. Ready for Week 1 topology.')

## Part 1: Metric Spaces on Geographic Data

A metric space (X, d) consists of a set X and a distance function d(x,y) satisfying:
- Non-negativity, symmetry, identity of indiscernibles
- Triangle inequality: d(x,z) <= d(x,y) + d(y,z)

For Chicago neighborhoods, we use Euclidean distance in projected coordinates (UTM Zone 16N).

In [ ]:
# Create synthetic Chicago tract centroids
np.random.seed(42)
n_tracts = 100

lat = np.random.uniform(41.65, 42.0, n_tracts)
lon = np.random.uniform(-87.9, -87.5, n_tracts)
tract_ids = [f'{i:05d}' for i in range(n_tracts)]

# Convert to UTM (meters) for accurate distances
from pyproj import Transformer
transformer = Transformer.from_crs('EPSG:4326', 'EPSG:32616')
x, y = transformer.transform(lat, lon)
coords_utm = np.column_stack([x, y])

# Compute distance matrix
D = distance_matrix(coords_utm, coords_utm)

print(f'Loaded {n_tracts} tracts')
print(f'Distance range: {D[D>0].min()/1000:.2f} to {D.max()/1000:.2f} km')

## Part 2: Connectedness of Geographic Networks

A graph is connected if there is a path between every pair of vertices.

At increasing distance thresholds, Chicago's tract network transitions from fragmented (many small clusters) to fully connected.

In [ ]:
# Build graphs at different radii and analyze connectivity
radii = np.arange(500, 15000, 500)
n_components = []

for r in radii:
    G = nx.Graph()
    for i in range(n_tracts):
        G.add_node(i)
    for i in range(n_tracts):
        for j in range(i+1, n_tracts):
            if D[i, j] <= r:
                G.add_edge(i, j)
    n_components.append(nx.number_connected_components(G))

# Plot connectivity
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(radii / 1000, n_components, 'o-', linewidth=2, markersize=6)
ax.set_xlabel('Edge radius (km)', fontsize=12)
ax.set_ylabel('Number of connected components', fontsize=12)
ax.set_title('Graph Fragmentation vs. Radius', fontsize=14)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/connectivity_analysis.png', dpi=150)
plt.show()

print('Connectivity analysis complete')

## Part 3: Multi-Scale Structure

The connectivity analysis reveals Chicago's natural multi-scale structure:

- **Local scale (0.5km):** Neighborhoods are mostly disconnected
- **District scale (2km):** Regional clustering emerges
- **City scale (5km+):** Fully connected

This structure explains why neural networks need depth: a shallow network's receptive field (~1-2km) cannot see patterns that emerge only at city scale (~5km+).

In [ ]:
print('\n' + '='*60)
print('WEEK 1 TOPOLOGY FOUNDATIONS: COMPLETE')
print('='*60)
print('\nKey findings:')
print('  • Geographic data has multi-scale structure')
print('  • Shallow GNNs cannot propagate information far enough')
print('  • Depth = necessary for expressivity on geographic problems')
print('\nNext: Week 2-3 will formalize with Stone-Weierstrass & GNN theory.')